# Hotel Booking Cancellation Prediction

### Project Summary

Hotels frequently face revenue uncertainty due to reservation cancellations. Accurately predicting which reservations are likely to cancel can help hotels improve revenue forecasting, optimize overbooking strategies, and better allocate resources.

The goal of this project is to develop a predictive model that identifies reservations likely to cancel using historical hotel booking data. This analysis follows the CRISP-DM framework, including data exploration, data cleaning, modeling, and evaluation. The resulting insights may help hotel managers make more informed decisions regarding booking policies and customer management.

In [3]:
import pandas as pd

# load dataset
df = pd.read_csv("../data/hotel_bookings.csv")

# preview the data
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Data & Data Dictionary

This dataset contains hotel booking records and includes reservation details, customer information, and whether the booking was canceled. The following cells preview the data structure and summarize the features included in the dataset.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

### Data Dictionary

| Feature | Data Type | Description |
|---|---|---|
| hotel | object | Type of hotel (Resort Hotel or City Hotel) |
| is_canceled | int64 | Whether the booking was canceled (1 = yes, 0 = no) |
| lead_time | int64 | Number of days between booking date and arrival date |
| arrival_date_year | int64 | Year of arrival |
| arrival_date_month | object | Month of arrival |
| arrival_date_week_number | int64 | Week number of arrival date |
| arrival_date_day_of_month | int64 | Day of the month of arrival |
| stays_in_weekend_nights | int64 | Number of weekend nights booked |
| stays_in_week_nights | int64 | Number of weekday nights booked |
| adults | int64 | Number of adults included in the booking |
| children | float64 | Number of children included in the booking |
| babies | int64 | Number of babies included in the booking |
| meal | object | Type of meal booked |
| country | object | Country of origin of the guest |
| market_segment | object | Market segment designation |
| distribution_channel | object | Booking distribution channel |
| is_repeated_guest | int64 | Whether the guest has booked before (1 = yes, 0 = no) |
| previous_cancellations | int64 | Number of previous bookings canceled by the customer |
| previous_bookings_not_canceled | int64 | Number of previous bookings not canceled by the customer |
| reserved_room_type | object | Code for room type originally reserved |
| assigned_room_type | object | Code for room type assigned at check-in |
| booking_changes | int64 | Number of changes made to the booking |
| deposit_type | object | Type of deposit made for the booking |
| agent | float64 | ID of the travel agent that made the booking |
| company | float64 | ID of the company/entity that made the booking |
| days_in_waiting_list | int64 | Number of days the booking was on the waiting list |
| customer_type | object | Type of customer |
| adr | float64 | Average daily rate |
| required_car_parking_spaces | int64 | Number of parking spaces required |
| total_of_special_requests | int64 | Number of special requests made by the customer |
| reservation_status | object | Reservation status at the end of the process |
| reservation_status_date | object | Date of the final reservation status |

## Selected Features for Exploratory Analysis

To limit the scope of exploratory analysis, four features were selected that are expected to have a strong relationship with reservation cancellations.

| Feature | Reason for Selection |
|-------|-------|
| lead_time | Longer booking lead times may increase the likelihood of cancellation |
| deposit_type | Deposit policies may influence customer commitment to reservations |
| previous_cancellations | Past customer behavior may indicate future cancellation risk |
| total_of_special_requests | Guests with more requests may be more committed to the reservation |

In [5]:
selected_features = [
    "lead_time",
    "deposit_type",
    "previous_cancellations",
    "total_of_special_requests",
    "is_canceled"
]

In [6]:
import pandas as pd

stats_rows = []

for col in selected_features:
    s = df[col]
    
    row = {
        "feature": col,
        "count_values": s.count(),
        "count_missing": s.isna().sum(),
        "unique_values": s.nunique(),
        "mode": s.mode().iloc[0] if not s.mode().empty else None
    }
    
    if pd.api.types.is_numeric_dtype(s):
        row.update({
            "min": s.min(),
            "Q1": s.quantile(0.25),
            "median": s.median(),
            "Q3": s.quantile(0.75),
            "max": s.max(),
            "mean": s.mean(),
            "std": s.std(),
            "skewness": s.skew(),
            "kurtosis": s.kurt()
        })
    else:
        row.update({
            "min": None,
            "Q1": None,
            "median": None,
            "Q3": None,
            "max": None,
            "mean": None,
            "std": None,
            "skewness": None,
            "kurtosis": None
        })
    
    stats_rows.append(row)

univariate_stats = pd.DataFrame(stats_rows)
univariate_stats

,feature,count_values,count_missing,unique_values,mode,min,Q1,median,Q3,max,mean,std,skewness,kurtosis
0,lead_time,119390,0,479,0,0.0,18.0,69.0,160.0,737.0,104.011416,106.863097,1.346550,1.696449
1,deposit_type,119390,0,3,No Deposit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,previous_cancellations,119390,0,15,0,0.0,0.0,0.0,0.0,26.0,0.087118,0.844336,24.458049,674.073693
3,total_of_special_requests,119390,0,6,0,0.0,0.0,0.0,1.0,5.0,0.571363,0.792798,1.349189,1.492565
4,is_canceled,119390,0,2,0,0.0,0.0,0.0,1.0,1.0,0.370416,0.482918,0.536678,-1.712005


In [7]:
univariate_stats.to_excel("../excel/hotel_univariate_statistics.xlsx", index=False)

## Numeric Relationships (Correlation)

In [8]:
import pandas as pd

numeric_features = [
    "lead_time",
    "previous_cancellations",
    "total_of_special_requests"
]

correlation_results = []

for col in numeric_features:
    corr = df[col].corr(df["is_canceled"])
    
    correlation_results.append({
        "feature": col,
        "pearson_correlation_with_is_canceled": corr
    })

correlation_df = pd.DataFrame(correlation_results)
correlation_df

,feature,pearson_correlation_with_is_canceled
0,lead_time,0.293123
1,previous_cancellations,0.110133
2,total_of_special_requests,-0.234658


## Categorical relationship (Chi-square)

In [9]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df["deposit_type"], df["is_canceled"])

chi2, p, dof, expected = chi2_contingency(contingency_table)

chi_square_result = pd.DataFrame({
    "feature": ["deposit_type"],
    "chi_square_statistic": [chi2],
    "p_value": [p]
})

chi_square_result

,feature,chi_square_statistic,p_value
0,deposit_type,27677.329241,0.0


In [10]:
bivariate_results = pd.concat([correlation_df, chi_square_result], ignore_index=True)
bivariate_results

,feature,pearson_correlation_with_is_canceled,chi_square_statistic,p_value
0,lead_time,0.293123,NaN,NaN
1,previous_cancellations,0.110133,NaN,NaN
2,total_of_special_requests,-0.234658,NaN,NaN
3,deposit_type,NaN,27677.329241,0.0


In [11]:
bivariate_results.to_excel("../excel/hotel_bivariate_statistics.xlsx", index=False)

## Data Cleaning & Preparation

Before modeling, the dataset needs to be checked for missing values, data types, and any basic quality issues that could affect model performance. In this project, I will review missing data and confirm that the selected features and label are in a usable format for machine learning.

In [12]:
# Check shape, missing values, and data types
print("Shape of dataset:", df.shape)
print("\nMissing values:")
print(df[selected_features + ["is_canceled"]].isnull().sum())

print("\nData types:")
print(df[selected_features + ["is_canceled"]].dtypes)

Shape of dataset: (119390, 32)

Missing values:
lead_time                    0
deposit_type                 0
previous_cancellations       0
total_of_special_requests    0
is_canceled                  0
is_canceled                  0
dtype: int64

Data types:
lead_time                    int64
deposit_type                   str
previous_cancellations       int64
total_of_special_requests    int64
is_canceled                  int64
is_canceled                  int64
dtype: object


### Cleaning Step 1: Check for Missing Values and Data Types

Before building predictive models, the dataset was examined for missing values and incorrect data types in the selected features and the target variable. Missing values can cause issues during model training and may bias results if not handled properly.

The selected features (`lead_time`, `deposit_type`, `previous_cancellations`, and `total_of_special_requests`) and the label (`is_canceled`) were checked for missing values and data types. The results show that there are no missing values in these variables. Therefore, no rows needed to be removed and no imputation was required.

### Cleaning Step 2: Encode Categorical Variables

Machine learning models require numeric inputs. However, the feature `deposit_type` is a categorical variable stored as text. To make this feature usable for modeling, it was converted into numeric variables using one-hot encoding.

One-hot encoding creates a binary column for each category of the variable. This allows the model to interpret each deposit type as a separate feature without introducing unintended numerical relationships between categories.

In [13]:
# Create modeling dataset with only features + label
model_df = df[selected_features].copy()
model_df["is_canceled"] = df["is_canceled"]

# One-hot encode deposit_type
model_df = pd.get_dummies(model_df, columns=["deposit_type"], drop_first=True)

# Preview dataset
model_df.head()

,lead_time,previous_cancellations,total_of_special_requests,is_canceled,deposit_type_Non Refund,deposit_type_Refundable
0,342,0,0,0,False,False
1,737,0,0,0,False,False
2,7,0,0,0,False,False
3,13,0,0,0,False,False
4,14,0,1,0,False,False


## Modeling & Evaluation

In this section, predictive models will be built to classify whether a hotel booking is canceled. The prepared dataset will be split into features and target, and multiple classification algorithms will be compared using cross-validation. Model performance will be evaluated using accuracy, precision, recall, F1 score, and AUC.

In [14]:
# Convert boolean columns to integers
bool_cols = model_df.select_dtypes(include="bool").columns
model_df[bool_cols] = model_df[bool_cols].astype(int)

# Define features and target
X = model_df.drop("is_canceled", axis=1)
y = model_df["is_canceled"]

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Feature columns:
['lead_time', 'previous_cancellations', 'total_of_special_requests', 'deposit_type_Non Refund', 'deposit_type_Refundable']

Target distribution:
is_canceled
0    75166
1    44224
Name: count, dtype: int64


### Train and Compare Multiple Models

Three classification algorithms will be trained and evaluated using cross-validation:

- Logistic Regression
- Decision Tree
- Random Forest

Cross-validation helps estimate model performance more reliably than a single train/test split.

In [15]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = {}

for name, model in models.items():
    
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=5,
        scoring=scoring
    )
    
    results[name] = {
        metric: cv_results[f"test_{metric}"].mean()
        for metric in scoring
    }

results

{'Logistic Regression': {'accuracy': np.float64(0.749593768322305),
  'precision': np.float64(0.8887449710617409),
  'recall': np.float64(0.3653849693922085),
  'f1': np.float64(0.5058627599800017),
  'roc_auc': np.float64(0.75411329548886)},
 'Decision Tree': {'accuracy': np.float64(0.7327665633637658),
  'precision': np.float64(0.7367540081328925),
  'recall': np.float64(0.41956395626063114),
  'f1': np.float64(0.5254463054176217),
  'roc_auc': np.float64(0.7462326506138219)},
 'Random Forest': {'accuracy': np.float64(0.7314850489990785),
  'precision': np.float64(0.7284046527749239),
  'recall': np.float64(0.4247647547758918),
  'f1': np.float64(0.5269268450293579),
  'roc_auc': np.float64(0.7546027161450627)}}

### Hyperparameter Tuning

Based on the cross-validation results, Random Forest performed best overall, so it was selected for hyperparameter tuning. GridSearchCV was used to test different combinations of parameters and identify the best-performing version of the model.

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

# Create base model
rf = RandomForestClassifier(random_state=42)

# Grid search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

# Fit grid search
grid_search.fit(X, y)

# Show best parameters and best score
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validated F1 score:")
print(grid_search.best_score_)

Best parameters:
{'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}

Best cross-validated F1 score:
0.5323693945194841


The grid search identified the best combination of Random Forest hyperparameters based on cross-validated F1 score. This tuned model was then selected as the final candidate model for further interpretation and prediction.

### Feature Selection

To identify the most influential predictors, feature importance was calculated using the tuned Random Forest model. This method estimates how much each feature contributes to the model's predictions. Features with higher importance values were considered more influential in predicting hotel booking cancellations.

This approach was used to confirm whether the selected features were meaningful predictors and to document the final feature set used in the model.

In [17]:
import pandas as pd

# Get the best tuned model
best_rf = grid_search.best_estimator_

# Extract feature importances
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance

,Feature,Importance
3,deposit_type_Non Refund,0.508830
0,lead_time,0.283591
1,previous_cancellations,0.115544
2,total_of_special_requests,0.090928
4,deposit_type_Refundable,0.001107


The feature importance results show which variables contributed most to predicting cancellations. Features with the highest importance were retained as the final predictors because they provided the most useful information to the model. Based on these results, the final model used `lead_time`, `previous_cancellations`, `total_of_special_requests`, and the encoded `deposit_type` categories as the final feature set.

## Final Model & Predictions

The tuned Random Forest model was selected as the final model based on its cross-validated performance. The model is demonstrated below by generating predictions for a small sample of booking records. These predictions estimate whether each reservation is likely to be canceled.

In [19]:
# Final model from grid search
final_model = grid_search.best_estimator_

# Take a small sample of bookings
sample_bookings = X.sample(5, random_state=42)

# Generate predictions
predictions = final_model.predict(sample_bookings)

# Combine results for viewing
prediction_results = sample_bookings.copy()
prediction_results["predicted_cancellation"] = predictions

prediction_results

,lead_time,previous_cancellations,total_of_special_requests,deposit_type_Non Refund,deposit_type_Refundable,predicted_cancellation
30946,203,0,0,0,0,0
40207,82,0,0,0,0,0
103708,25,0,1,0,0,0
85144,1,0,0,0,0,0
109991,70,0,0,0,0,0


## Inferences

The exploratory analysis and predictive modeling identified several key patterns related to hotel booking cancellations. The Random Forest model showed that deposit policies, lead time, previous cancellations, and special requests are important predictors of whether a booking will be canceled.

Feature importance results indicated that the most influential variable was whether the booking required a non-refundable deposit. Bookings with non-refundable deposits were significantly less likely to be canceled. Lead time was also a strong predictor, with bookings made far in advance showing higher cancellation risk. Additionally, guests with a history of previous cancellations were more likely to cancel future bookings. Finally, guests who made more special requests tended to be slightly less likely to cancel, suggesting higher commitment to their reservation.

## Implications

These findings suggest that certain booking characteristics can help hotels identify reservations that carry a higher risk of cancellation. Understanding these patterns allows hotel managers to make more informed operational and revenue management decisions.

For example, bookings made far in advance may require additional monitoring because they have a greater chance of being canceled. Similarly, guests with a history of cancellations may represent a higher-risk customer segment. Deposit policies also appear to play a critical role in influencing guest behavior, as non-refundable deposits significantly reduce cancellation likelihood.

While the model provides useful insights, it is important to recognize that predictions are probabilistic rather than certain. External factors such as travel disruptions, personal circumstances, or economic conditions may still affect guest behavior.

## Business Recommendations

Based on the results of the analysis and predictive modeling, several actionable recommendations can be made:

1. Encourage non-refundable deposits for reservations with long lead times. Since deposit policies strongly influence cancellation behavior, requiring deposits for early bookings could reduce cancellation rates.

2. Implement cancellation risk monitoring for bookings made far in advance. Reservations with long lead times could be flagged by the predictive model so that hotels can adjust inventory or overbooking strategies accordingly.

3. Track and manage repeat cancellers. Guests with previous cancellation histories may represent higher risk, and hotels may consider stricter deposit policies or confirmation reminders for these customers.

4. Use predictive modeling to support revenue management decisions. Integrating a cancellation prediction model into reservation systems could help hotels anticipate cancellation patterns and optimize room availability.

These strategies could help hotels reduce revenue loss from cancellations and improve operational planning.